# Preprocessing

Packages and settings

In [ ]:
# Public libraries
import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
from pathlib import Path
import re
import math
import time
import functools
from tqdm import tqdm

def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.coverage_functions import coverage_calculator, plot_time_spacing, plot_time_series, calculate_coverage, identify_time_gaps, plot_time_gaps
from tools.labeling_functions import fully_relabel_and_consolidate, relabel_items, rename_items, rename_items_by_modifications, plot_dish_time_series

# Prevent needing to restart the kernel to reflect changes in other files
%load_ext autoreload
%autoreload 2

# Set query engine
original_query = pd.DataFrame.query
#pd.DataFrame.query = functools.partialmethod(pd.DataFrame.query, engine="python")

Import data

In [ ]:
# Before
if not 'VLZX7K2M9QD4T_before.parquet' in os.listdir("data/1_palate_data_parquet/orders_item_level"):
    data_before = pd.read_excel('VLZX7K2M9QD4T_data.xlsx', sheet_name='4.18.21-10.17.21 Pre BSF')
else:
    data_before = pd.read_parquet("data/1_palate_data_parquet/orders_item_level/VLZX7K2M9QD4T_before.parquet")

# After
if not 'VLZX7K2M9QD4T_after.parquet' in os.listdir("data/1_palate_data_parquet/orders_item_level"):
    data_after = pd.read_excel('VLZX7K2M9QD4T_data.xlsx', sheet_name='10.18.21-4.17.22 After BSF')
else:
    data_after = pd.read_parquet("data/1_palate_data_parquet/orders_item_level/VLZX7K2M9QD4T_after.parquet")
    
# Check Excel results for unique categories and number NA
# data_before.astype(str).apply(lambda s: s.str.lower().nunique())
# data_after.astype(str).apply(lambda s: s.str.lower().nunique())

# All
data = pd.concat([data_before, data_after]).reset_index(drop=True)
promo_datetime = pd.to_datetime('10-18-21').tz_localize('UTC')
before_after_details_true = pd.concat([pd.read_csv('data/before_after_details_true.csv'), 
                                       pd.DataFrame([{'location_id': 'VLZX7K2M9QD4T',
                                                      'cross_over_date': promo_datetime, 
                                                      'first_plant_based_mention': 'Black Sheep', 
                                                      'batch': np.nan, 
                                                      'promo_name': 'Black Sheep'}])]).set_index('location_id', drop=True)
before_after_details_true.to_csv('data/4_palate_data_parquet_relabeled/before_after_details_true.csv')
locations_true = pd.concat([pd.read_parquet('data/2_palate_data_parquet_cleaned/locations.parquet'),
                            pd.DataFrame([{'location_id':'VLZX7K2M9QD4T',
                                          'cuisine':'Greek',
                                          'restaurant_type': 'Fast causal',
                                          'pos_type':'Square'}])]).set_index('location_id', drop=True)
locations_true.to_csv('data/4_palate_data_parquet_relabeled/locations.csv', index=True)

## Inspection

In [ ]:
print(f"Data dimensions: {data.shape[0]} rows x {data.shape[1]} cols \n")
print(f"Unique items: {data['Item'].unique().size}; Unique transactions/orders: {data['Transaction ID'].unique().size} \n")
print(f"Data types and unique categories: \n")
print(pd.merge(data.dtypes.to_frame('Data Types'), 
         data.apply(lambda s: s.nunique()).to_frame('# Categories'), 
         left_index=True, 
         right_index=True)
       .rename_axis(columns='Variables')
       .to_string())

pd.concat([data.head(1), data.tail(1)])

## Cleaning

In [ ]:
# Function to handle the conversion to string while keeping NaNs
def to_string_or_na(series):
    return series.where(series.isna(), series.astype(str))

# Create undercase snakecase column mapping
new_column_names = data.columns.str.lower().str.replace(" ", "_")
column_mapping = pd.Series(new_column_names, index=data.columns).to_dict()
standard_column_names = {'item':'item_name',
                         'qty':'item_quantity',
                         'modifiers_applied':'item_modifications',
                         'category':'dish_category',
                         'net_sales':'item_price'}

# Cleaning pipeline
df = (data
      .rename(columns = column_mapping)
      .pipe(lambda df: print(f"Number of NA dates: {df['date'].isna().sum()}") or df) # Diagnostics
      .pipe(lambda df: print(f"Number of NA items: {df['item'].isna().sum()}") or df) # Diagnostics
      .pipe(lambda df: print(f"Number of NA categories: {df['category'].isna().sum()}") or df) # Diagnostics
      .pipe(lambda df: print(f"Number of duplicate rows: {df.duplicated().sum()}") or df) # Diagnostics
      .fillna({'category': "Other"})
      .astype({'date': str}) # Convert to string since there are no NAs
      .pipe(lambda df: df.apply(lambda col: to_string_or_na(col) if col.dtype in ['object'] else col)) # Make numeric entries into strings
      .assign(created_at = lambda df: pd.to_datetime(df['date'] + ' ' + df['time'],).dt.tz_localize('UTC'))
      .set_index('created_at', drop=True)
      .drop(columns=['date', 'time'])
      .rename(columns = standard_column_names)
      .assign(is_plant_based = "Unsure",
              item_price = lambda df: df['item_price'] * 100,
              unit_price = lambda df: df['item_price'] / df['item_quantity'],
              item_name = lambda df: df['item_name'].str.title(),
              item_modifications = lambda df: df['item_modifications'].str.title(),
              location_id = 'VLZX7K2M9QD4T')
      .sort_values(['created_at', 'item_name'])
      .assign()
      .pipe(lambda df: print(f"Number of rows: {df.shape[0]}") or df)
      .query('event_type == "Payment"')
      .pipe(lambda df: print(f"Number of rows: {df.shape[0]}") or df)
      .query('dining_option != "To Go"')
      .pipe(lambda df: print(f"Number of rows: {df.shape[0]}") or df)
      #.astype({col: 'category' for col in category_columns}) # Change to category columns
      )

print(f"Expected Nonzero Days: {(df.index[[-1]] - df.index[[0]]).days[0]}, Actual Nonzero Days: {df.resample('D')['item_quantity'].sum().to_frame('col').query('0 < col').size}")

# Category columns
categories_threshold = 300
limited_categories = (df
                      .apply(lambda s: s.nunique())
                      .to_frame(name='num_categories')
                      .query('num_categories < @categories_threshold')
                      .index
                      .tolist()
                      )
string_categories = (df
                     .dtypes
                     .to_frame(name='dtype')
                     .astype(str)
                     .query('dtype == "object"')
                     .index
                     .tolist()
                     )
category_columns = set(limited_categories).intersection(set(string_categories))
print(category_columns)

df.to_parquet('data/2_palate_data_parquet_cleaned/orders_item_level/VLZX7K2M9QD4T.parquet', index=True)

pd.concat([df.head(1), df.tail(1)])

## Dish Name and Category Consolidation

In [ ]:
print(df_relabeled.query('item_name.str.contains("Pota")'))

In [ ]:
nonfood = [
    "Utensils",
    "Tip the Staff",
    "Delivery Fee - Taxable",
    "Commissary - Packaging",
    "Commissary - Operating Supplies",
    "VLZX7K2M9QD4T Retail Items",
    "Commissary - Retail Goods",
    "Commissary - Chemicals (NOPA)",
    "Commissary - Chemicals (Valencia)",
    "Gift Card",
    "Catering Service Fees",
    "Venue Fee",
    "Other"
    ]

ingredients = [
    "Commissary - Bulk Grocery Ingredients",
    "Commissary - Produce Ingredients"
    ]

commissary = [
    "Commissary - Meat",
    "Commissary - Dairy",
    "Commissary - Prepped Veg",
    "Commissary - Pita",
    "Commissary - Smallwares",
    ]

drinks = [
    "Beverages",
    "Wine",
    "Beer",
    "Retail Greek Wines",
    "Commissary - Beverage",
    "Commissary - Wine",
    "Commissary - Beer",
    "Brunch - Food" # just coffee
    ]

elsewhere = [
    "Delta Airlines",
    "Catering",
    "Gate Gourmet Delivery", # delivery service
    "CAVIAR", # bay area delivery service
    "*Caviar Week: World Tour - Prix Fixe Menu", # presumably tied to CAVIAR delivery service
    "Goldbelly Food" # meal kits, delivery service
    ]

specials = [
    "Specials" # during late 2021 to early 2022: either 1) entirely takeout or if dine in 2) not on the official menu
    ]

special_events = [
    "8th Anniversary Specials",
    ]

# To be counted as a menu item, it needs its own entry and needs to be more than $2
sides = ["Extra Yogurt Sauce",
         "Side Of Yogurt Sauce",
         "Side VLZX7K2M9QD4T Hot Sauce",
         "Hot Sauce Bottle",
         "Extra Topping"]

category_remove_list = (nonfood + 
                        ingredients + 
                        drinks + 
                        elsewhere + 
                        commissary + 
                        specials #+
                        #+ special_events (seems to be dine in commemorating the intro of a new location)
                        )

modification_name_changes_1 = [
    ('Black Sheep Sandwich', 'Pork|Chicken|Add Lamb', 'Meat Black Sheep Sandwich'),
    ('Black Sheep Salad', 'Pork|Chicken|Add Lamb', 'Meat Black Sheep Salad'),
    ('Black Sheep Sandwich', 'No Cheese' , 'No Cheese Black Sheep Sandwich'),
    ('Black Sheep Salad', 'No Cheese' , 'No Cheese Black Sheep Salad'),
    
    ('Veg Sandwich', 'Pork|Chicken|Add Lamb', 'Meat Veg Sandwich'),
    ('Veg Salad', 'Pork|Chicken|Add Lamb', 'Meat Veg Salad'),
    ('Veg Sandwich', 'No Cheese' , 'No Cheese Veg Sandwich'),
    ('Veg Salad', 'No Cheese' , 'No Cheese Veg Salad'),
    
    ('VLZX7K2M9QD4T Side Green Salad', 'Pork|Chicken|Add Lamb', 'Meat VLZX7K2M9QD4T Side Green Salad'),
    ('Greek Fries', 'Pork|Chicken|Add Lamb', 'Meat Greek Fries'),
    ('VLZX7K2M9QD4T Side Green Salad', 'No Yogurt' , 'No Yogurt VLZX7K2M9QD4T Side Green Salad'),
    ('Greek Fries', 'No Yogurt' , 'No Yogurt Greek Fries'),
    ('VLZX7K2M9QD4T Side Green Salad', 'Yogurt', 'Yogurt VLZX7K2M9QD4T Side Green Salad'),
    ('Greek Fries', 'Yogurt' , 'Yogurt Greek Fries')
]

modification_name_changes_2 = [
    ('No Cheese Black Sheep Sandwich', 'No Yogurt' , 'Vegan Black Sheep Sandwich'),
    ('No Cheese Black Sheep Salad', 'No Yogurt' , 'Vegan Black Sheep Salad'),
    ('No Cheese Black Sheep Sandwich', '', 'Black Sheep Sandwich'),
    ('No Cheese Black Sheep Salad', '', 'Black Sheep Salad'),
    
    ('No Cheese Veg Sandwich', 'No Yogurt' , 'Vegan Veg Sandwich'),
    ('No Cheese Veg Salad', 'No Yogurt' , 'Vegan Veg Salad'),
    ('No Cheese Veg Sandwich', '', 'Veg Sandwich'),
    ('No Cheese Veg Salad', '', 'Veg Salad'),
    
    ('No Yogurt VLZX7K2M9QD4T Side Green Salad', 'No Cheese', 'Vegan VLZX7K2M9QD4T Side Green Salad'),
    ('No Yogurt Greek Fries', 'No Cheese', 'Vegan Greek Fries'),
    ('No Yogurt VLZX7K2M9QD4T Side Green Salad', '' , 'Vegetarian VLZX7K2M9QD4T Side Green Salad'),
    ('No Yogurt Greek Fries', '' , 'Vegetarian Greek Fries'),
    ('Yogurt VLZX7K2M9QD4T Side Green Salad', '' , 'Vegetarian VLZX7K2M9QD4T Side Green Salad'),
    ('Yogurt Greek Fries', '' , 'Vegetarian Greek Fries'),
    ('VLZX7K2M9QD4T Side Green Salad', 'No Cheese' , 'Vegan VLZX7K2M9QD4T Side Green Salad'),
    ('Greek Fries', 'No Cheese' , 'Vegan Greek Fries'),
]

meat_list = [
    "Chicken Sandwich",
    "Chicken Salad",
    "Pork Sandwich",
    "Pork Salad",
    "Lamb Sandwich",
    "Lamb Salad",
    "Avgolemono Soup", # Contains Chicken
    "Juicy Potatoes",
    "Side Of Meat",
    "Side Of Meat (6 Oz.)",
    "Side Of Meat (3 Oz.)",
    "Whole Feta-Brined Rotisserie Chicken",
    "Roasted Chicken Meal" ,
    "Stuffed Breakfast Pita",
    "Pastitsio",
    "Meat Black Sheep Sandwich",
    "Meat Black Sheep Salad",
    "Meat Veg Sandwich",
    "Meat Veg Salad",
    "Meat VLZX7K2M9QD4T Side Green Salad",
    "Meat Greek Fries",
]

vegetarian_list = [
    "Veg Sandwich", # Contains Mizithra Cheese & Garlic Yogurt
    "Veg Salad", # Contains Mizithra Cheese & Garlic Yogurt
    "Greek Fries", # Contains Mizithra Cheese
    "VLZX7K2M9QD4T Side Green Salad", # Contains Feta Cheese
    "Plain Frozen Greek Yogurt", # Contains Dairy
    "Sour Cherry Syrup Frozen Yogurt", # Contains Dairy
    "Greek Olive Oil & Flaky Sea Salt Frozen Yogurt", # Contains Dairy
    "Baklava Crumbles & Honey Syrup Frozen Yogurt", # Contains Dairy
    "Cretan Wildflower Honey Frozen Yogurt", # Contains Dairy
    "Extra Yogurt Sauce", # Contains Dairy
    "Side Of Yogurt Sauce", # Contains Dairy
    "Black Sheep Sandwich", # Contains Dairy
    "Black Sheep Salad", # Contains Dairy
    "Rainbow Surprise!", # Contains Dairy
    "Tsoureki & Whipped Manouri", # Contains Dairy
    "Straus Organic Greek Yogurt", # Contains Dairy
    "Tzatziki", # Contains Dairy
    "Vegetarian VLZX7K2M9QD4T Side Green Salad", # Contains Feta Cheese
    "Vegetarian Greek Fries", # Contains Mizithra Cheese
]

vegan_list = [
    "Pita Bread", # Typically vegan
    "Melitzanosalata",
    "Vegan Black Sheep Sandwich",
    "Vegan Black Sheep Salad",
    "Vegan Veg Sandwich",
    "Vegan Veg Salad",
    "Vegan VLZX7K2M9QD4T Side Green Salad",
    "Vegan Greek Fries",
]

dish_names = {
    # Sandwiches
    "Black Sheep Sandwich" : ["Black Sheep Lamb Sandwich","Meat Black Sheep Sandwich","Vegan Black Sheep Sandwich"], # seems to just be unclear coding for the first couple weeks
    "Black Sheep Salad" : ["Black Sheep Lamb Salad","Meat Black Sheep Salad","Vegan Black Sheep Salad"], # seems to just be unclear coding for the first couple weeks
    
    "Veg Sandwich" : ["Vegetarian Veg Sandwich","Meat Veg Sandwich","Vegan Veg Sandwich"],
    "Veg Salad" : ["Vegetarian Veg Salad","Meat Veg Salad","Vegan Veg Salad"], 
    
    "Greek Fries" : ["Vegetarian Greek Fries","Meat Greek Fries","Vegan Greek Fries"], 
    "VLZX7K2M9QD4T Side Green Salad" : ["Vegetarian VLZX7K2M9QD4T Side Green Salad","Meat VLZX7K2M9QD4T Side Green Salad","Vegan VLZX7K2M9QD4T Side Green Salad"], 
    
    # Sides
    #"Side Of Meat": ["Side Of Meat (6 Oz.)", "Side Of Meat (3 Oz.)"],
}

rare_list = df['item_name'].value_counts().to_frame(name='counts').query('counts <= 5').index.tolist() # doesn't make a difference (all with few sales are nonfood etc.)

df_relabeled = (df
              .pipe(lambda df: print(f"Number of rows: {df.shape[0]}") or df)
              .query('~dish_category.isin(@category_remove_list)', local_dict={'category_remove_list': category_remove_list})
              .pipe(lambda df: print(f"Number of rows: {df.shape[0]}") or df)
              .pipe(fully_relabel_and_consolidate,
                    remove = rare_list + sides,
                    name_changes = dish_names)
              .pipe(lambda df: print(f"Number of rows: {df.shape[0]}") or df)
              .pipe(rename_items_by_modifications, modification_name_changes=modification_name_changes_1)
              .pipe(rename_items_by_modifications, modification_name_changes=modification_name_changes_2)
              .pipe(relabel_items, vegan_list=vegan_list, vegetarian_list=vegetarian_list, meat_list=meat_list)
              )

df_relabeled.to_parquet('data/4_palate_data_parquet_relabeled/relabeled/VLZX7K2M9QD4T.parquet', index=True)

df_consolidated = df_relabeled.pipe(rename_items, name_changes = dish_names)

plot_dish_time_series(df_consolidated, 'VLZX7K2M9QD4T', before_after_details_true, scale=1000)

df_consolidated.to_parquet('data/4_palate_data_parquet_relabeled/consolidated/VLZX7K2M9QD4T.parquet', index=True)

In [ ]:
# Useful checks

# labeled = vegan_list + vegetarian_list + meat_list
# df_cleaned.query('item_name.isin(@vegan_list)')['item_name'].value_counts()

# df.loc[df.query('dish_category.str.contains("Anniv")').index[0]:]

# print(df_cleaned.query('~dish_category.isin(@remove_list)')#.query('dish_category == "Specials"')
#  [['item_name','dining_option']].value_counts(dropna=False).to_string())

# print(df
#       .query('item_name == "Meal For Two"')
#       ['item_modifications']
#       .str.replace('Side Hot Sauce, ','')
#       .str.replace('Greek Fries, ','')
#       .str.replace('Side Hot Sauce', '')
#       .str.split(', ')
#       .apply(lambda l: l[0])
#       .value_counts()
#       .to_string())
# print()
# print(df
#       .query('item_name == "Meal For Two"')
#       ['item_modifications']
#       .str.replace('Side Hot Sauce, ','')
#       .str.replace('Greek Fries, ','')
#       .str.replace('Side Hot Sauce', '')
#       .str.split(', ')
#       .apply(lambda l: l[1])
#       .value_counts()
#       .to_string())
# print()
# print(df
#       .query('item_name == "Meal For Two"')
#       ['item_modifications']
#       .str.replace('Side Hot Sauce, ','')
#       .str.replace('Greek Fries, ','')
#       .str.replace('Greek Fries|Side Hot Sauce|GREEK STYLE|3oz. |6oz. |Add ','', regex=True)
#       .str.split(', ')
#       .apply(lambda l: l[2] if len(l)>2 else None)
#       .value_counts()
#       .to_string())
# print()
# print(df
#       .query('item_name == "Meal For Two"')
#       ['item_modifications']
#       .str.replace('Side Hot Sauce, ','')
#       .str.replace('Greek Fries, ','')
#       .str.replace('Greek Fries|Side Hot Sauce|GREEK STYLE|3oz. |6oz. |Add ','', regex=True)
#       .str.split(', ')
#       .apply(lambda l: l[3] if len(l)>3 else None)
#       .value_counts()
#       .to_string())

## Integrity Checks

In [ ]:
plot_time_series(df, promo_datetime, subset=False) # 12/13/2021 unknown missing day, 4 days for christmas
plt.show()
plot_time_series(df, promo_datetime, subset=True) # 11/25/21 Thanksgiving
plt.show()
plot_time_series(df.query('item_name.str.contains("Black Sheep")'), promo_datetime, subset=False)
plt.show()


General coverage statistics

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction

coverages = calculate_coverage(df, {'promo':'Black Sheep', 'intro_date':promo_datetime}, transaction_id="transaction_id")
coverage_info = pd.DataFrame([(key, val) for key, val in coverages.items()], columns=['metric', 'coverage'])
coverage_info = (coverage_info
                 .assign(period = lambda df: df['metric'].apply(lambda x: x.split('_')[0]),
                                     freq = lambda df: df['metric'].apply(lambda x: '_'.join(x.split('_')[1:])))
                 .pivot(index='period', columns='freq', values='coverage')
                 .loc[['bef','aft','all','b2m','a2m','4mo']]
                 [['W-M_cover', 'D_cover', '12H_cover', '6H_cover']])

display(coverage_info)

if 'time_differences_details' not in locals() or 'time_differences' not in locals():
    VLZX7K2M9QD4T_time_differences, VLZX7K2M9QD4T_time_differences_details = identify_time_gaps(df, quantity_col='item_quantity', index_name='created_at')
    time_differences, time_differences_details = VLZX7K2M9QD4T_time_differences, VLZX7K2M9QD4T_time_differences_details
plot_time_gaps(time_differences, colorbar_max=75)

Investigate

In [ ]:
nonzero_gap_details = (time_differences_details
                       .dropna()
                       .astype(int)
                       .rename_axis(['day', 'date', 'datetime'])
                       .to_frame('gap')
                       .query('gap != 0'))

display(nonzero_gap_details.query('gap >= 3'))

# Seven hour gap
display(pd.concat([df.loc[:'2021-10-18 08:00:55'].tail(2), df.loc['2021-10-18 08:00:55':].head(0)]))

# Two hour gap
display(pd.concat([df.loc[:'2021-04-19 10:01:03'].tail(2), df.loc['2021-04-19 10:01:03':].head(0)]))

## Special Events and Sales

In [ ]:
plot_time_series(df.query('dish_category.str.contains("Commissary")'), promo_datetime, subset=False)
plt.show()
plot_time_series(df.query('dish_category.str.contains("Delta")'), promo_datetime, subset=False)
plt.show()
plot_time_series(df.query('dish_category.str.contains("Catering")'), promo_datetime, subset=False)
plt.show()

## Takeaways

Upsides
- No missingness 
- Non food items are categorized
- Clear promotional item
- Clearly categorized items according to restaurant function (airline, commissary, catering, etc.)
- No blank items
- Limited encoding errors and idiosyncracy in item names

Downsides
- No data before the promo introduction
- No plant-based labeling
- Potential duplication